# 🔍 Interactive Model Inspector & Diagnostic Evaluation Tool
### Clinical Prediction, Probability Breakdown & Performance Metrics for Monkeypox Detection

This interactive tool provides a unified diagnostic system for supervisors, researchers, and clinicians to:
1. **Select any trained model** (from the 8 Vision Transformers and 4 Hybrid CNN-Transformers: CoAtNet, CvT, Visformer, ConViT).
2. **Predict on custom images** (single image or folder) OR **sample random images** from the held-out test split.
3. **Inspect prediction probabilities** across all 3 classes (`Chickenpox`, `Monkeypox`, `Normal`) with visual bar charts.
4. **Overlay labels directly on the images** with color-coded diagnostic badges (Green = Correct, Red = Misclassified).
5. **Compute quantitative metrics** (Accuracy, Balanced Acc, Macro-F1, Precision, Recall, Confusion Matrix) when ground truth is available.

## 1. Connect to Google Drive & Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Monkeypox_Final')
    print("Google Drive mounted successfully!")
except Exception as e:
    PROJECT_DIR = Path(os.getcwd())
    print(f"Note: Running locally or already mounted. Base: {PROJECT_DIR}")

# Validate PROJECT_DIR
if not PROJECT_DIR.exists():
    my_drive = Path('/content/drive/MyDrive')
    candidates = list(my_drive.glob('*[Mm]onkeypox*')) if my_drive.exists() else []
    if candidates:
        PROJECT_DIR = candidates[0]
        print(f"Auto-located project directory at: {PROJECT_DIR}")

# 2. Locate Test Directory (supports mkpox_dataset_grayscale, processed, dataset)
def locate_test_dir(base_dir):
    candidates = [
        base_dir / 'mkpox_dataset_grayscale' / 'test',
        base_dir / 'mkpox_dataset_grayscale' / 'mkpox_dataset_grayscale' / 'test',
        Path('/content/dataset_local/test'),
        base_dir / 'processed' / 'test',
        base_dir / 'dataset' / 'test',
        base_dir / 'test'
    ]
    for c in candidates:
        if c.exists() and any(c.iterdir()):
            return c
    for p in base_dir.rglob('test'):
        if p.is_dir() and any('pox' in s.name.lower() or 'norm' in s.name.lower() for s in p.iterdir() if s.is_dir()):
            return p
    return None

TEST_DIR = locate_test_dir(PROJECT_DIR)
print(f"Project Directory:  {PROJECT_DIR}")
print(f"Test Directory:     {TEST_DIR} (Ready: {TEST_DIR is not None})")


## 2. Load Core Prediction Engine & Dependencies

In [ ]:
# Install dependencies
!pip install -q timm torchmetrics scikit-learn seaborn matplotlib pandas numpy pillow ipywidgets

import json
import random
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import transforms
import timm
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix
)

# Canonical classes & standard transforms
CANONICAL_CLASSES = ['Chickenpox', 'Monkeypox', 'Normal']
IMAGE_SIZE = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# ==========================================================================
# MONKEYPOX PREDICTION ENGINE (Self-Contained)
# ==========================================================================
class MonkeypoxPredictor:
    """
    Unified Inference & Evaluation Engine for Dermatological Image Classification.
    Supports ViT, DeiT, Swin, BEiT, MaxViT, PiT, MAE, FlexiViT, CoAtNet, CvT, Visformer, ConViT.
    """
    def __init__(
        self,
        checkpoint_path: Optional[str] = None,
        model_name: Optional[str] = None,
        device: Optional[str] = None
    ):
        self.device = torch.device(device if device else ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path else None
        self.model_name = model_name
        self.class_names = CANONICAL_CLASSES
        self.model = None
        self.model_identifier = None
        self._load_model()

    def _load_model(self):
        checkpoint_dict = None
        if self.checkpoint_path and self.checkpoint_path.exists():
            print(f"Loading checkpoint: {self.checkpoint_path}...")
            checkpoint_dict = torch.load(self.checkpoint_path, map_location=self.device, weights_only=False)
            if 'class_names' in checkpoint_dict:
                self.class_names = checkpoint_dict['class_names']
            if 'config' in checkpoint_dict and 'model_identifier' in checkpoint_dict['config']:
                self.model_identifier = checkpoint_dict['config']['model_identifier']

        if not self.model_identifier:
            self.model_identifier = self.model_name if self.model_name else 'vit_base_patch16_224'

        print(f"Instantiating architecture: {self.model_identifier} (Classes: {self.class_names})...")
        try:
            self.model = timm.create_model(self.model_identifier, pretrained=True if checkpoint_dict is None else False, num_classes=len(self.class_names))
        except Exception as e:
            print(f"Direct creation failed ({e}). Trying fallback vit_base_patch16_224...")
            self.model = timm.create_model('vit_base_patch16_224', pretrained=True if checkpoint_dict is None else False, num_classes=len(self.class_names))

        if checkpoint_dict and 'model_state_dict' in checkpoint_dict:
            self.model.load_state_dict(checkpoint_dict['model_state_dict'])
            print(f"Successfully loaded weights from Epoch {checkpoint_dict.get('epoch', 'N/A')}!")
        elif checkpoint_dict is None:
            print("Using ImageNet pretrained weights with fresh classification head.")

        self.model.to(self.device)
        self.model.eval()

    def predict_single(self, image_path: Path, use_tta: bool = False) -> Dict[str, Any]:
        with Image.open(image_path) as raw_img:
            raw_img_rgb = raw_img.convert('RGB')
            tensor_img = eval_transforms(raw_img_rgb).unsqueeze(0).to(self.device)

        with torch.no_grad():
            out_orig = self.model(tensor_img)
            p_orig = torch.softmax(out_orig, dim=1)

            if use_tta:
                tensor_flipped = torch.flip(tensor_img, dims=[-1])
                out_flip = self.model(tensor_flipped)
                p_flip = torch.softmax(out_flip, dim=1)
                probs = (0.5 * (p_orig + p_flip)).cpu().numpy()[0]
            else:
                probs = p_orig.cpu().numpy()[0]

        pred_idx = int(np.argmax(probs))
        pred_label = self.class_names[pred_idx]
        conf = float(probs[pred_idx])

        # Infer true label if image is inside a class folder
        true_label = None
        parent_name = image_path.parent.name
        if parent_name in self.class_names:
            true_label = parent_name
        elif any(c.lower() in image_path.stem.lower() for c in self.class_names):
            for c in self.class_names:
                if c.lower() in image_path.stem.lower():
                    true_label = c
                    break

        return {
            'image_path': str(image_path),
            'filename': image_path.name,
            'true_label': true_label,
            'predicted_label': pred_label,
            'confidence': conf,
            'is_correct': (true_label == pred_label) if true_label else None,
            'probabilities': {self.class_names[i]: float(probs[i]) for i in range(len(self.class_names))}
        }

    def predict_batch(self, image_paths: List[Path], use_tta: bool = False) -> List[Dict[str, Any]]:
        return [self.predict_single(p, use_tta=use_tta) for p in image_paths]

# ==========================================================================
# SAMPLING & VISUALIZATION UTILITIES
# ==========================================================================
def sample_random_test_images(test_dir: Path, num_samples: int = 6) -> List[Path]:
    all_test_images = []
    for c in CANONICAL_CLASSES:
        cdir = test_dir / c
        if cdir.exists():
            files = [f for f in cdir.glob('*') if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png'} and not f.name.startswith('.')]
            all_test_images.extend(files)
    if not all_test_images:
        for f in test_dir.rglob('*'):
            if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png'} and not f.name.startswith('.'):
                all_test_images.append(f)
    if not all_test_images:
        raise FileNotFoundError(f"No test images found in: {test_dir}")
    num_samples = min(num_samples, len(all_test_images))
    return random.sample(all_test_images, num_samples)

def render_prediction_visuals(predictions: List[Dict[str, Any]], output_path: Path, title_suffix: str = ""):
    n = len(predictions)
    cols = min(3, n)
    rows = int(np.ceil(n / cols))

    fig = plt.figure(figsize=(6.5 * cols, 4.5 * rows), dpi=200)
    plt.suptitle(f"Model Inference & Probability Analysis {title_suffix}", fontsize=16, y=0.99)

    for idx, p in enumerate(predictions):
        img_p = Path(p['image_path'])
        true_lbl = p['true_label']
        pred_lbl = p['predicted_label']
        conf = p['confidence'] * 100.0
        probs = p['probabilities']
        is_corr = p['is_correct']

        if is_corr is True:
            badge_color = '#2ca02c' # Green
        elif is_corr is False:
            badge_color = '#d62728' # Red
        else:
            badge_color = '#1f77b4' # Blue

        # 1. Image Subplot
        ax_img = plt.subplot2grid((rows, cols * 2), (idx // cols, (idx % cols) * 2))
        try:
            with Image.open(img_p) as img:
                ax_img.imshow(img.convert('RGB'))
        except Exception:
            ax_img.text(0.5, 0.5, 'Image Load Error', ha='center', va='center')
        ax_img.axis('off')

        badge_str = f"Pred: {pred_lbl} ({conf:.1f}%)\nTrue: {true_lbl if true_lbl else 'Unknown'}"
        ax_img.set_title(badge_str, fontsize=11, fontweight='bold', color='white', backgroundcolor=badge_color, pad=6)

        # 2. Probability Bar Chart
        ax_bar = plt.subplot2grid((rows, cols * 2), (idx // cols, (idx % cols) * 2 + 1))
        classes = list(probs.keys())
        prob_vals = [probs[c] * 100.0 for c in classes]
        bar_colors = ['#2ca02c' if c == pred_lbl else '#aec7e8' for c in classes]

        bars = ax_bar.barh(classes, prob_vals, color=bar_colors, edgecolor='black', height=0.55)
        ax_bar.set_xlim(0, 105)
        ax_bar.set_xlabel('Probability (%)', fontsize=10)
        ax_bar.grid(axis='x', linestyle='--', alpha=0.5)

        for bar in bars:
            w = bar.get_width()
            ax_bar.text(w + 1.5, bar.get_y() + bar.get_height() / 2, f"{w:.1f}%", va='center', ha='left', fontsize=9, fontweight='bold')
        ax_bar.tick_params(axis='y', labelsize=10)

    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, bbox_inches='tight', dpi=200)
    plt.close()
    print(f"Saved prediction visualization to: {output_path}")

def compute_evaluation_metrics(predictions: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    has_ground_truth = all(p['true_label'] is not None for p in predictions)
    if not has_ground_truth:
        return None
    y_true = [p['true_label'] for p in predictions]
    y_pred = [p['predicted_label'] for p in predictions]
    return {
        'total_samples': len(predictions),
        'accuracy': round(accuracy_score(y_true, y_pred) * 100.0, 2),
        'balanced_accuracy': round(balanced_accuracy_score(y_true, y_pred) * 100.0, 2),
        'macro_f1': round(f1_score(y_true, y_pred, average='macro', zero_division=0) * 100.0, 2),
        'precision': round(precision_score(y_true, y_pred, average='macro', zero_division=0) * 100.0, 2),
        'recall': round(recall_score(y_true, y_pred, average='macro', zero_division=0) * 100.0, 2),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=CANONICAL_CLASSES).tolist(),
        'classes': CANONICAL_CLASSES
    }

print("Interactive Model Inspector Engine Loaded Successfully!")


## 3. Interactive Selection & Configuration
Select the model architecture, input mode (Random Test Sampling vs Custom Images), number of images, and inference options below:

In [ ]:
#@title 🎛️ Diagnostic Configuration Form

# Model Selection
model_choice = "11_CoAtNet_Hybrid" #@param ["01_ViT_Base", "02_DeiT_Base", "03_Swin_Base", "04_BEiT_Base", "05_MaxViT_Base", "06_PiT_Base", "07_MAE_Base", "08_FlexiViT_Base", "11_CoAtNet_Hybrid", "12_CvT_Hybrid", "13_Visformer_Hybrid", "14_ConViT_Hybrid"]
custom_checkpoint_path = "" #@param {type:"string"}

# Input Mode
input_mode = "Random Test Samples" #@param ["Random Test Samples", "Custom Image Path/Folder"]
num_random_samples = 6 #@param {type:"slider", min:1, max:24, step:1}
custom_image_input = "" #@param {type:"string"}

# Inference Protocol
use_tta = True #@param {type:"boolean"}

print(f"Selected Model:          {model_choice}")
print(f"Input Mode:              {input_mode}")
if input_mode == 'Random Test Samples':
    print(f"Number of Samples:       {num_random_samples}")
else:
    print(f"Custom Input:            {custom_image_input}")
print(f"Test-Time Augmentation:  {use_tta}")


## 4. Run Inference, Probability Analysis & Visual Inspection

In [ ]:
# 1. Resolve Checkpoint Path
ckpt_path = None
if custom_checkpoint_path.strip():
    ckpt_path = Path(custom_checkpoint_path.strip())
else:
    default_ckpt = PROJECT_DIR / 'results' / model_choice / 'checkpoints' / 'best_model.pth'
    if default_ckpt.exists():
        ckpt_path = default_ckpt

# Model identifier mapping
model_id_map = {
    "01_ViT_Base": "vit_base_patch16_224",
    "02_DeiT_Base": "deit_base_patch16_224",
    "03_Swin_Base": "swin_base_patch4_window7_224",
    "04_BEiT_Base": "beit_base_patch16_224",
    "05_MaxViT_Base": "maxvit_base_tf_224",
    "06_PiT_Base": "pit_b_224",
    "07_MAE_Base": "vit_base_patch16_224.mae",
    "08_FlexiViT_Base": "flexivit_base.patch16_in21k",
    "11_CoAtNet_Hybrid": "coatnet_1_rw_224.sw_in1k",
    "12_CvT_Hybrid": "cvt_13",
    "13_Visformer_Hybrid": "visformer_small",
    "14_ConViT_Hybrid": "convit_small"
}

selected_id = model_id_map.get(model_choice, "vit_base_patch16_224")
predictor = MonkeypoxPredictor(
    checkpoint_path=str(ckpt_path) if ckpt_path else None,
    model_name=selected_id
)

# 2. Select Images
eval_images = []
if input_mode == "Random Test Samples":
    if not TEST_DIR or not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory not found. Please verify TEST_DIR path in Section 1.")
    eval_images = sample_random_test_images(TEST_DIR, num_samples=num_random_samples)
else:
    cpath = Path(custom_image_input.strip())
    if cpath.is_dir():
        eval_images = [f for f in cpath.glob('*') if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png'}][:num_random_samples]
    elif cpath.is_file():
        eval_images = [cpath]
    else:
        raise FileNotFoundError(f"Custom path does not exist: {cpath}")

# 3. Run Inference
print(f"\nEvaluating {len(eval_images)} images with {model_choice} (TTA={use_tta})...")
predictions = predictor.predict_batch(eval_images, use_tta=use_tta)

# 4. Display Results Table
df_res = pd.DataFrame([
    {
        'Filename': p['filename'],
        'True Class': p['true_label'] if p['true_label'] else 'Unknown',
        'Predicted': p['predicted_label'],
        'Confidence': f"{p['confidence']*100:.1f}%",
        'Chickenpox (%)': f"{p['probabilities']['Chickenpox']*100:.1f}%",
        'Monkeypox (%)': f"{p['probabilities']['Monkeypox']*100:.1f}%",
        'Normal (%)': f"{p['probabilities']['Normal']*100:.1f}%",
        'Status': ('CORRECT' if p['is_correct'] else 'INCORRECT') if p['is_correct'] is not None else 'N/A'
    }
    for p in predictions
])

print("="*85)
print("INFERENCE RESULTS TABLE")
print("="*85)
display(df_res)

# 5. Render Visual Diagnostic Cards with Probabilities
pred_fig_path = PROJECT_DIR / 'results' / 'interactive_predictions.png'
render_prediction_visuals(predictions, pred_fig_path, title_suffix=f"({model_choice})")

# Display figure inline in Colab
from IPython.display import Image as IPImage, display as ipydisplay
ipydisplay(IPImage(filename=str(pred_fig_path)))

# 6. Calculate Quantitative Metrics (if ground truth available)
metrics = compute_evaluation_metrics(predictions)
if metrics:
    print("\n" + "="*85)
    print("SAMPLE SET PERFORMANCE METRICS")
    print("="*85)
    print(f"Accuracy:          {metrics['accuracy']:.2f}%")
    print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.2f}%")
    print(f"Macro-F1:          {metrics['macro_f1']:.2f}%")
    print(f"Macro-Precision:   {metrics['precision']:.2f}%")
    print(f"Macro-Recall:      {metrics['recall']:.2f}%")
    
    # Confusion matrix heatmap
    import seaborn as sns
    plt.figure(figsize=(5, 4))
    sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=CANONICAL_CLASSES, yticklabels=CANONICAL_CLASSES)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Sample Confusion Matrix')
    plt.tight_layout()
    plt.show()
